<a href="https://colab.research.google.com/github/Wadsonsp/multiobjective-ids-generalization/blob/main/notebooks/pipeline_experimentos_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
!pip install kaggle -q

In [6]:
import os

# Cole aqui seus dados do kaggle.json
os.environ['KAGGLE_USERNAME'] = 'wadson'
os.environ['KAGGLE_KEY'] = 'KGAT_f00c6f12f3b88b1305ecbb7decdbb3ee'


In [7]:
import zipfile
import os

pasta_destino = "/content/drive/MyDrive/mestrado/Datasets"
os.makedirs(pasta_destino, exist_ok=True )

# Baixar NF-UNSW-NB15-v2 (~442 MB)
print("Baixando NF-UNSW-NB15-v2...")
!kaggle datasets download -d sankurisrinath/nf-unsw-nb15-v2csv -p /content/temp1
with zipfile.ZipFile("/content/temp1/nf-unsw-nb15-v2csv.zip", 'r') as z:
    z.extractall(pasta_destino)
print("✅ NF-UNSW-NB15-v2 extraído!")

# Baixar NF-ToN-IoT-v2
print("\nBaixando NF-ToN-IoT-v2...")
!kaggle datasets download -d shahidabbas76/nf-ton-iot-v2-full-dataset -p /content/temp2
with zipfile.ZipFile("/content/temp2/nf-ton-iot-v2-full-dataset.zip", 'r') as z:
    z.extractall(pasta_destino)
print("✅ NF-ToN-IoT-v2 extraído!")


Baixando NF-UNSW-NB15-v2...
Dataset URL: https://www.kaggle.com/datasets/sankurisrinath/nf-unsw-nb15-v2csv
License(s): DbCL-1.0
100% 35.3M/35.3M [00:00<00:00, 87.7MB/s]

✅ NF-UNSW-NB15-v2 extraído!

Baixando NF-ToN-IoT-v2...
Dataset URL: https://www.kaggle.com/datasets/shahidabbas76/nf-ton-iot-v2-full-dataset
License(s): other
100% 177M/177M [00:01<00:00, 110MB/s]

✅ NF-ToN-IoT-v2 extraído!


In [8]:
print("\nArquivos na pasta Datasets:")
for arquivo in os.listdir(pasta_destino):
    caminho = os.path.join(pasta_destino, arquivo)
    tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)
    print(f"  {arquivo}  →  {tamanho_mb:.2f} MB")



Arquivos na pasta Datasets:
  NF-UNSW-NB15-v2.csv  →  421.40 MB
  NF-ToN-IoT-v2.csv  →  2552.50 MB


In [10]:
import pandas as pd, numpy as np

origem  = "/content/drive/MyDrive/mestrado/Datasets/NF-ToN-IoT-v2.csv"
destino = "/content/drive/MyDrive/mestrado/Datasets/NF-ToN-IoT-v2-2M.csv"
N_ALVO, TOTAL_OFICIAL = 2_000_000, 16_940_496

# 1) Contagem total lendo só a coluna Attack (barato em memória)
total = sum(len(c) for c in pd.read_csv(origem, usecols=["Attack"], chunksize=2_000_000))
print(f"Fluxos no arquivo: {total:,} | oficial: {TOTAL_OFICIAL:,} | "
      f"{'INTEGRO' if total == TOTAL_OFICIAL else 'DIVERGENTE - conferir fonte!'}")

# 2) Amostra estratificada de 2M lendo em blocos de 1M linhas
frac = N_ALVO / total
rng, primeiro = np.random.RandomState(42), True
for chunk in pd.read_csv(origem, chunksize=1_000_000, low_memory=False):
    amostra = chunk.groupby("Attack", group_keys=False).apply(
        lambda g: g.sample(frac=min(1.0, frac), random_state=rng))
    amostra.to_csv(destino, mode="w" if primeiro else "a", header=primeiro, index=False)
    primeiro = False

n = sum(len(c) for c in pd.read_csv(destino, usecols=["Attack"], chunksize=2_000_000))
print(f"Amostra salva no Drive: {n:,} fluxos")

Fluxos no arquivo: 16,940,496 | oficial: 16,940,496 | INTEGRO


/tmp/ipykernel_1393/4127398926.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  amostra = chunk.groupby("Attack", group_keys=False).apply(
/tmp/ipykernel_1393/4127398926.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  amostra = chunk.groupby("Attack", group_keys=False).apply(
/tmp/ipykernel_1393/4127398926.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This beha

Amostra salva no Drive: 1,999,994 fluxos


In [11]:
%cd /content/multiobjective-ids-generalization
!sed -i 's/arquivo: "NF-ToN-IoT-v2.csv"/arquivo: "NF-ToN-IoT-v2-2M.csv"/' src/config.yaml
!sed -i 's/amostra_estratificada: 2000000/amostra_estratificada: null/' src/config.yaml
!grep -A3 "NF-ToN-IoT-v2:" src/config.yaml

/content/multiobjective-ids-generalization
    NF-ToN-IoT-v2:
      arquivo: "NF-ToN-IoT-v2-2M.csv"
      url_oficial: "https://rdm.uq.edu.au/files/a4ad7080-ef9c-11ed-a964-b70596e96ad5"
      # amostra estratificada de 2,0 milhões de fluxos para equiparar volume


In [12]:
!python src/algoritmo2_avaliacao.py --mascara cheia --amostra 50000

[config] h=xgboost | k(x)=37 de d=37
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
Traceback (most recent call last):
  File "/content/multiobjective-ids-generalization/src/algoritmo2_avaliacao.py", line 134, in <module>
    main()
  File "/content/multiobjective-ids-generalization/src/algoritmo2_avaliacao.py", line 107, in main
    criterios = avaliar_fitness(
                ^^^^^^^^^^^^^^^^
  File "/content/multiobjective-ids-generalization/Modulos/avaliacao.py", line 182, in avaliar_fitness
    nome: avaliar_intra_dataset(mascara, X, y, nome_clf, cv_folds, seed)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/m

In [13]:
%cd /content/multiobjective-ids-generalization
!git pull
!python src/algoritmo2_avaliacao.py --mascara cheia --amostra 50000

/content/multiobjective-ids-generalization
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 2.37 KiB | 606.00 KiB/s, done.
From https://github.com/Wadsonsp/multiobjective-ids-generalization
   dc59860..c9e3207  main       -> origin/main
Updating dc59860..c9e3207
Fast-forward
 Modulos/avaliacao.py    | 30 +++++++++++++++++++++----
 tests/test_avaliacao.py | 58 +++++++++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 84 insertions(+), 4 deletions(-)
[config] h=xgboost | k(x)=37 de d=37
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y ha